## From the top: parse → embed → tier → trace_pairs → Neo4j → checkpoint

In [ ]:
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

TRAIN_JSONL = ROOT / "train.jsonl"
MAX_LINES = 1090
MAX_REQUIREMENTS = 2000
DOC_ID = "train_jsonl"
CHECKPOINT_PATH = ROOT / "requirements_checkpoint_embed_tier.json"

### 1. Parse requirements from train.jsonl

In [ ]:
from src.schema import Entry

def is_req_value(v):
    if v is True:
        return True
    if v is False or v is None:
        return False
    return str(v).strip().lower() in ("true", "yes", "1", "requirement")

requirements = []
lines_read = 0
with open(TRAIN_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if lines_read >= MAX_LINES:
            break
        line = line.strip()
        if not line:
            continue
        try:
            row = json.loads(line)
            out = row.get("output") or {}
            for e in out.get("entries") or []:
                if not is_req_value(e.get("is_requirement")):
                    continue
                requirements.append(
                    Entry(
                        doc_title=e.get("doc_title") or "",
                        page_number=int(e.get("page_number", 0)),
                        section_title=e.get("section_title") or "",
                        text=(e.get("text") or "").strip(),
                        is_requirement="true",
                    )
                )
                if len(requirements) >= MAX_REQUIREMENTS:
                    break
        except (json.JSONDecodeError, KeyError, TypeError):
            continue
        lines_read += 1
        if len(requirements) >= MAX_REQUIREMENTS:
            break

print(f"Parsed {len(requirements)} requirements from {TRAIN_JSONL}")

### 2. Embed requirements (Titan)

In [ ]:
from src.titan_embeddings import get_embeddings_for_entries, embeddings_cache_path

embed_cache = embeddings_cache_path(ROOT, DOC_ID)
embeddings = get_embeddings_for_entries(requirements, embed_cache, DOC_ID, dimensions=1024)
print(f"Embeddings: {len(embeddings)} vectors")

### 3. Assign tiers (plant / system / component)

In [ ]:
from src.tier_by_similarity import assign_tiers_by_similarity_with_fallback

requirements_with_tiers = assign_tiers_by_similarity_with_fallback(
    requirements, embeddings, sample_size=60
)
for e, tier in requirements_with_tiers[:8]:
    print(f"  [{tier}] {e.text[:60]}...")
print(f"... total {len(requirements_with_tiers)}")

### 4. Trace pairs

In [ ]:
from src.trace_similarity_nova import get_trace_pairs_single_choice_nova_with_fallback

trace_pairs = get_trace_pairs_single_choice_nova_with_fallback(
    requirements_with_tiers, embeddings, k=5, nova_batch_size=5
)
print(f"Trace pairs (one per system/component): {len(trace_pairs)}")

### 5. Neo4j

In [ ]:
from src.neo4j_loader import (
    get_driver,
    load_tiered_requirements_into_neo4j,
    requirement_full_id,
    delete_trace_edges,
    create_trace_edges,
    filter_trace_pairs_to_adjacent_tiers,
)

driver = get_driver()
n = load_tiered_requirements_into_neo4j(
    requirements_with_tiers,
    driver=driver,
    document_id=DOC_ID,
    embeddings=embeddings if embeddings else None,
)
print(f"Loaded {n} requirement nodes.")

full_ids = [requirement_full_id(DOC_ID, e, i) for i, (e, _) in enumerate(requirements_with_tiers)]
deleted = delete_trace_edges(driver)
if deleted:
    print(f"Deleted {deleted} existing TRACES_TO edges.")
all_pairs = filter_trace_pairs_to_adjacent_tiers(requirements_with_tiers, trace_pairs)
edges = create_trace_edges(full_ids, all_pairs, driver=driver)
print(f"Created {edges} TRACES_TO edges ({len(all_pairs)} pairs).")
driver.close()

### 6. Save checkpoint

In [ ]:
checkpoint = {
    "requirements_with_tiers": [{"entry": e.model_dump(), "tier": t} for e, t in requirements_with_tiers],
    "trace_pairs": [list(p) for p in trace_pairs],
    "doc_id": DOC_ID,
}
if embeddings and len(embeddings) == len(requirements_with_tiers):
    checkpoint["embeddings"] = embeddings
CHECKPOINT_PATH.write_text(json.dumps(checkpoint, indent=2), encoding="utf-8")
print(f"Saved checkpoint: {len(requirements_with_tiers)} requirements, {len(trace_pairs)} pairs.")